In [1]:
# test_rag_menu.py
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

# Configuración
MODEL_PATH = "/home/alan_dev/Documentos/projects/casalaguna_agentic/backend/data/embeddings/model/models--intfloat--multilingual-e5-large/snapshots/0dc5580a448e4284468b8909bae50fa925907bc5"
VECTOR_DB_PATH = "/home/alan_dev/Documentos/projects/casalaguna_agentic/backend/data/embeddings/vector_db"
COLLECTION_NAME = "casalaguna_rag"

def test_rag_system():
    print("🧪 Probando sistema RAG ESPECÍFICO DE MENÚ...")
    
    # 1. Cargar modelo
    print("🔹 Cargando modelo...")
    model = SentenceTransformer(MODEL_PATH)
    
    # 2. Conectar a ChromaDB
    print("🔹 Conectando a ChromaDB...")
    client = chromadb.PersistentClient(
        path=VECTOR_DB_PATH,
        settings=Settings(anonymized_telemetry=False)
    )
    collection = client.get_collection(COLLECTION_NAME)
    
    print(f"📊 Documentos en la colección: {collection.count()}")
    
    # 3. CONSULTAS ESPECÍFICAS DEL MENÚ (basadas en el JSON que me mostraste)
    test_queries = [
        # Consultas sobre categorías específicas
        "¿Qué tostadas tienen en el menú?",
        "¿Qué opciones de tacos marinos ofrecen?",
        "¿Tienen aguachiles en el menú?",
        "¿Qué tipos de camarones preparan?",
        "¿Ofrecen opciones vegetarianas en Casa Laguna?",
        "¿Qué postres tienen disponibles?",
        "¿Tienen menú para niños?",
        
        # Consultas sobre platillos específicos
        "¿Qué es la Tostada de camarón en aguachile verde?",
        "¿Cuánto cuesta la Torre de frutos del mar?",
        "¿Tienen opciones de pescado a la parrilla?",
        "¿Qué incluye la Parrillada Mar y Tierra?",
        "¿Tienen pasta con mariscos?",
        "¿Qué hamburguesas ofrecen?",
        "¿Tienen opciones sin gluten?",
        
        # Consultas sobre precios y características
        "¿Cuál es el platillo más picante del menú?",
        "¿Qué platillos tienen precio alrededor de $300?",
        "¿Tienen opciones de mar y tierra combinadas?",
        "¿Qué incluye el menú de mariscos?",
        "¿Ofrecen ceviche de pescado?"
    ]
    
    for query in test_queries:
        print(f"\n{'='*80}")
        print(f"🔍 Consulta: '{query}'")
        print('='*80)
        
        # Generar embedding de la consulta
        query_text = f"query: {query}"
        query_embedding = model.encode(
            query_text,
            normalize_embeddings=True,
            show_progress_bar=False
        )
        
        # Buscar documentos similares
        results = collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=5,  # Aumentar a 5 para ver más resultados
            include=["documents", "metadatas", "distances"]
        )
        
        if results['documents'][0]:
            print(f"📈 Resultados encontrados: {len(results['documents'][0])}")
            for i, (doc, metadata, distance) in enumerate(zip(
                results['documents'][0], 
                results['metadatas'][0], 
                results['distances'][0]
            )):
                print(f"\n   📌 Resultado #{i+1}:")
                print(f"   📐 Similitud: {1-distance:.4f}" if distance < 2 else f"   📐 Distancia: {distance:.4f}")
                print(f"   🏷️  Categoría: {metadata.get('category', 'N/A')}")
                print(f"   📋 Tipo: {metadata.get('type', 'N/A')}")
                print(f"   📍 Ciudad: {metadata.get('city', 'N/A')}")
                print(f"   📄 Contenido (primeras 300 chars):\n   {doc[:300]}...")
                
                # Verificar si es del menú
                if metadata.get('type') == 'menu' or metadata.get('category') == 'menu':
                    print(f"   ✅ ¡ES DEL MENÚ!")
        else:
            print("   ⚠️ No se encontraron resultados para esta consulta.")
    
    print(f"\n{'='*80}")
    print("✅ Prueba de menú completada!")
    print('='*80)
    
    # 4. VERIFICACIÓN ESPECÍFICA: Buscar documentos de tipo 'menu'
    print(f"\n🔍 BUSCANDO DOCUMENTOS ESPECÍFICAMENTE DEL MENÚ:")
    
    # Buscar por tipo
    menu_docs = collection.get(where={"type": "menu"})
    print(f"   Documentos con type='menu': {len(menu_docs['ids'])}")
    
    # Buscar por categoría
    menu_category_docs = collection.get(where={"category": "menu"})
    print(f"   Documentos con category='menu': {len(menu_category_docs['ids'])}")
    
    # Buscar por cualquier mención de "menu" en los documentos
    all_docs = collection.get()
    menu_related = []
    for i, doc in enumerate(all_docs['documents']):
        if 'menu' in doc.lower() or 'platillo' in doc.lower() or 'tostada' in doc.lower():
            menu_related.append({
                'id': all_docs['ids'][i],
                'type': all_docs['metadatas'][i].get('type', 'N/A'),
                'category': all_docs['metadatas'][i].get('category', 'N/A')
            })
    
    print(f"   Documentos relacionados con menú (búsqueda textual): {len(menu_related)}")
    if menu_related:
        print(f"   Ejemplos: {menu_related[:3]}")
    
    # 5. MOSTRAR ALGUNOS DOCUMENTOS PARA VER SU ESTRUCTURA
    print(f"\n🔍 MUESTRA DE DOCUMENTOS (primeros 5):")
    sample = collection.peek()
    for i in range(min(5, len(sample['ids']))):
        print(f"\n   Documento {i+1}:")
        print(f"   ID: {sample['ids'][i]}")
        print(f"   Tipo: {sample['metadatas'][i].get('type', 'N/A')}")
        print(f"   Categoría: {sample['metadatas'][i].get('category', 'N/A')}")
        print(f"   Contenido (primeros 200 chars): {sample['documents'][i][:200]}...")

if __name__ == "__main__":
    test_rag_system()

/home/alan_dev/miniconda3/envs/chatbot/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🧪 Probando sistema RAG ESPECÍFICO DE MENÚ...
🔹 Cargando modelo...
🔹 Conectando a ChromaDB...
📊 Documentos en la colección: 39

🔍 Consulta: '¿Qué tostadas tienen en el menú?'
📈 Resultados encontrados: 5

   📌 Resultado #1:
   📐 Similitud: 0.8177
   🏷️  Categoría: menu
   📋 Tipo: faq
   📍 Ciudad: N/A
   📄 Contenido (primeras 300 chars):
   passage: Pregunta frecuente: ¿Cuentan con opciones sin gluten en el menú? | Respuesta: : El menú incluye opciones sin gluten....
   ✅ ¡ES DEL MENÚ!

   📌 Resultado #2:
   📐 Similitud: 0.8159
   🏷️  Categoría: menu
   📋 Tipo: faq
   📍 Ciudad: N/A
   📄 Contenido (primeras 300 chars):
   passage: Pregunta frecuente: ¿Cuentan con opciones veganas en el menú? | Respuesta: : El menú incluye opciones veganas....
   ✅ ¡ES DEL MENÚ!

   📌 Resultado #3:
   📐 Similitud: 0.8151
   🏷️  Categoría: menu
   📋 Tipo: faq
   📍 Ciudad: N/A
   📄 Contenido (primeras 300 chars):
   passage: Pregunta frecuente: ¿Cuentan con opciones keto en el menú? | Respuesta: : El menú inc